# Iteration 3: attitude bias in the final position (precision at height)

**Loss targeted:** precision. About 20-25 % of answers at 300 m and above land beyond 1 sigma (6 or 3 points instead of 10).

**Hypothesis.** Patch v0.5.6 (revised) fitted a lateral angle of about +0.98 degrees on 80 answered photos of an older run (fused coarse, K 100).
An angle offset moves the nadir point by h x tan(angle): 1 degree at 350 m is 6 m, about 0.7 sigma, which is exactly the size of the high-altitude
misses. The fine fit is a 2D similarity of the north-up canvas and cannot see roll or pitch errors, so the bias passes straight to the answer.
It was never applied in `src/` (settings.GEOMETRY has no roll offset) and never checked on the current pipeline.

**Test (offline).** Signed errors of today's answers (rich dumps: chosen x, y and truth; yaw, pitch, roll per photo), model
e = h R(yaw) (a_f, a_r) + h R(yaw) (k_p tan pitch, k_r tan roll) + b (block `train/checks/pose_bias.py`), fitted on answers within 4 sigma.
Held out by **flight** (photos of one flight share a height, so folds are grouped by height): fit on 4/5 of the flights, correct the other 1/5.
NUDT photos have no attitude (pitch = roll = 0), so only the constant terms can be checked there.

In [1]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
pd.set_option("display.width", 220)
from train.checks.replay import load_set, decide_current, points, score, paired_bootstrap
from train.checks.pose_bias import fit, apply
from sklearn.model_selection import GroupKFold
DUMPS = ROOT / "tests/Test_Iteration_6/results/rich_dump"
SET_NAMES = {"official": "official", "nudt101": "nudt_seed101", "nudt202": "nudt_seed202", "nudt404": "nudt_fresh404"}
frames = {}
for name, folder in SET_NAMES.items():
    if not (DUMPS / folder).exists():
        continue
    rows = []
    for pid, ph in load_set(DUMPS / folder).items():
        r = decide_current(ph)
        rows.append(dict(photo=pid, answered=r["answered"], stage=r["stage"], ex=r["x"] - ph["truth"][0], ey=r["y"] - ph["truth"][1],
                         err_sigma=r["err_sigma"], sigma=ph["sigma"], h=ph["height"], yaw=ph["yaw"], pitch=ph["pitch"], roll=ph["roll"], points=r["points"],
                         q_heading=ph["queries"][r["chosen_owner"]]["heading"] if r["chosen_owner"] >= 0 else 0.0,
                         q_scale=ph["queries"][r["chosen_owner"]]["scale"] if r["chosen_owner"] >= 0 else 1.0))
    frames[name] = pd.DataFrame(rows)
    frames[name]["flight"] = frames[name].h.round(0)
print({k: (len(v), v.flight.nunique()) for k, v in frames.items()})
if "official" in frames:
    print(frames["official"][["pitch", "roll", "yaw", "h"]].describe().round(2))

{'official': (379, 67), 'nudt101': (300, 58), 'nudt202': (200, 48), 'nudt404': (200, 45)}
        pitch    roll     yaw       h
count  379.00  379.00  379.00  379.00
mean    -6.13    0.59  189.90  209.87
std      5.97    3.17   97.26   91.72
min    -15.81  -10.03   16.73  100.00
25%    -11.29   -0.63  105.62  125.85
50%     -6.53    0.34  180.44  150.50
75%     -1.12    1.35  272.51  299.90
max      9.22   15.41  353.98  350.70


## 1. Fit on all answered official photos (in-sample, to see the size of the terms)

In [2]:
if "official" in frames:
    ans = frames["official"][frames["official"].answered]
    for terms in (("const",), ("angle",), ("angle", "const"), ("tilt",), ("angle", "tilt", "const")):
        m = fit(ans, terms)
        print("%-28s mirror=%s  rms %.2f -> %.2f m   coef %s" % ("+".join(terms), m["mirror"], m["rms_before"], m["rms_after"],
              np.array2string(m["coef"], precision=5)))
    m = fit(ans, ("angle", "const"))
    print("angle terms in degrees (forward, lateral):", np.degrees(np.arctan(m["coef"][:2])).round(3))

const                        mirror=False  rms 4.01 -> 3.82 m   coef [-0.11874 -1.72699]
angle                        mirror=False  rms 4.01 -> 2.78 m   coef [0.01758 0.00178]
angle+const                  mirror=False  rms 4.01 -> 2.58 m   coef [ 0.01718  0.00181 -0.13994 -1.48034]
tilt                         mirror=False  rms 4.01 -> 3.66 m   coef [-0.07046  0.0039 ]
angle+tilt+const             mirror=False  rms 4.01 -> 2.56 m   coef [ 0.0185   0.00167  0.01392  0.01155 -0.19405 -1.45487]
angle terms in degrees (forward, lateral): [0.984 0.104]


## 2. Held out by flight: points before and after the correction

In [3]:
def cv_gain(df, terms, folds=5):
    df = df.reset_index(drop=True)
    new_points = df.points.to_numpy().copy(); new_err = df.err_sigma.to_numpy().copy()
    n_groups = df.flight.nunique()
    for tr, te in GroupKFold(n_splits=min(folds, n_groups)).split(df, groups=df.flight):
        train = df.iloc[tr]; train = train[train.answered]
        if (train.err_sigma <= 4).sum() < 10:
            continue
        m = fit(train, terms)
        test = df.iloc[te]
        ex, ey = apply(test, m)
        err = np.hypot(ex, ey) / test.sigma.to_numpy()
        new_err[te] = err
        new_points[te] = [points(e) if a else 0 for e, a in zip(err, test.answered)]
    return new_points, new_err
rows = []
for name, df in frames.items():
    for terms in (("const",), ("angle",), ("angle", "const"), ("angle", "tilt", "const")):
        if name != "official" and "tilt" in terms:
            continue
        newp, newe = cv_gain(df, terms)
        g = paired_bootstrap(df.points.to_numpy(), newp)
        a = df.answered.to_numpy()
        rows.append(dict(set=name, terms="+".join(terms), score_before=round(score(df.points), 2), score_after=round(score(newp), 2),
                         gain="%+.2f [%+.2f, %+.2f]" % g, wrong_before=int((df.points < 0).sum()), wrong_after=int((newp < 0).sum()),
                         within_1sigma_before=round((df.err_sigma[a] <= 1).mean(), 3), within_1sigma_after=round((newe[a] <= 1).mean(), 3),
                         median_err_before=round(np.median(df.err_sigma[a]), 3), median_err_after=round(np.median(newe[a]), 3)))
cv = pd.DataFrame(rows); display(cv)
cv.to_csv(ROOT / "tests/Test_Iteration_6/results/it3_pose_bias_cv.csv", index=False)

,set,terms,score_before,score_after,gain,wrong_before,wrong_after,within_1sigma_before,within_1sigma_after,median_err_before,median_err_after
0,official,const,53.54,53.22,"-0.32 [-0.82, +0.19]",0,0,0.951,0.936,0.564,0.542
1,official,angle,53.54,54.11,"+0.57 [+0.06, +1.08]",0,0,0.951,0.977,0.564,0.377
2,official,angle+const,53.54,54.36,"+0.82 [+0.38, +1.33]",0,0,0.951,0.988,0.564,0.328
3,official,angle+tilt+const,53.54,54.36,"+0.82 [+0.38, +1.33]",0,0,0.951,0.988,0.564,0.330
4,nudt101,const,55.56,55.56,"+0.00 [+0.00, +0.00]",0,0,0.989,0.989,0.378,0.382
5,nudt101,angle,55.56,55.64,"+0.08 [+0.00, +0.24]",0,0,0.989,0.993,0.378,0.199
6,nudt101,angle+const,55.56,55.64,"+0.08 [+0.00, +0.24]",0,0,0.989,0.993,0.378,0.195
7,nudt202,const,57.36,57.36,"+0.00 [+0.00, +0.00]",0,0,0.984,0.984,0.425,0.427
8,nudt202,angle,57.36,57.48,"+0.12 [-0.24, +0.48]",0,0,0.984,0.990,0.425,0.223
9,nudt202,angle+const,57.36,57.48,"+0.12 [-0.24, +0.48]",0,0,0.984,0.990,0.425,0.232


## 3. By height band (official, best terms, held out by flight)

In [4]:
df = frames["official"].reset_index(drop=True)
best_terms = ("angle", "const")
newp, newe = cv_gain(df, best_terms)
df["new_points"], df["new_err"] = newp, newe
df["band"] = pd.cut(df.h, [0, 150, 250, 400], labels=["<=150", "150-250", "250-400"])
a = df[df.answered]
display(a.groupby("band", observed=True).agg(answered=("points", "size"), score_before=("points", lambda s: round(s.mean() * 0.6 * 10 / 10, 2)),
        within1_before=("err_sigma", lambda s: round((s <= 1).mean(), 3)), within1_after=("new_err", lambda s: round((s <= 1).mean(), 3)),
        median_before=("err_sigma", "median"), median_after=("new_err", "median")).round(3))

,answered,score_before,within1_before,within1_after,median_before,median_after
band,,,,,,
<=150,142,5.95,0.979,0.993,0.415,0.306
150-250,58,5.88,0.948,1.000,0.536,0.306
250-400,145,5.82,0.924,0.979,0.751,0.372


## 4. Learned position refinement (idea B2)

**Input:** height, sin/cos of yaw, pitch, roll, the winning query's heading offset and scale factor. **Output:** the residual offset (east, south) in
metres, as a fraction of the height. **Training data:** answered official photos within 4 sigma, held out by flight. **Replaces:** nothing; it
corrects the answer. **Leakage risk:** the target is the truth offset, so evaluation is strictly out-of-flight. A small gradient-boosting
regressor against the linear model of section 2.

In [5]:
from sklearn.ensemble import GradientBoostingRegressor
def learned_cv(df, folds=5):
    df = df.reset_index(drop=True)
    X = np.stack([df.h / 300, np.sin(np.radians(df.yaw)), np.cos(np.radians(df.yaw)), df.pitch, df.roll, df.q_heading, df.q_scale], axis=1)
    new_points = df.points.to_numpy().copy(); new_err = df.err_sigma.to_numpy().copy()
    for tr, te in GroupKFold(n_splits=min(folds, df.flight.nunique())).split(df, groups=df.flight):
        train = df.iloc[tr]; keep = (train.answered & (train.err_sigma <= 4)).to_numpy()
        if keep.sum() < 20:
            continue
        pred = np.zeros((len(te), 2))
        for j, col in enumerate(("ex", "ey")):
            m = GradientBoostingRegressor(max_depth=2, n_estimators=150, learning_rate=0.05, subsample=0.8, random_state=0)
            m.fit(X[tr][keep], (train[col] / train.h).to_numpy()[keep])
            pred[:, j] = m.predict(X[te]) * df.h.to_numpy()[te]
        test = df.iloc[te]
        err = np.hypot(test.ex.to_numpy() - pred[:, 0], test.ey.to_numpy() - pred[:, 1]) / test.sigma.to_numpy()
        new_err[te] = err; new_points[te] = [points(e) if a else 0 for e, a in zip(err, test.answered)]
    return new_points, new_err
rows = []
for name, df in frames.items():
    newp, newe = learned_cv(df)
    g = paired_bootstrap(df.points.to_numpy(), newp); a = df.answered.to_numpy()
    rows.append(dict(set=name, model="GBM residual", score_before=round(score(df.points), 2), score_after=round(score(newp), 2), gain="%+.2f [%+.2f, %+.2f]" % g,
                     wrong_after=int((newp < 0).sum()), within_1sigma_before=round((df.err_sigma[a] <= 1).mean(), 3), within_1sigma_after=round((newe[a] <= 1).mean(), 3)))
learned = pd.DataFrame(rows); display(learned)
learned.to_csv(ROOT / "tests/Test_Iteration_6/results/it3_learned_refinement_cv.csv", index=False)

,set,model,score_before,score_after,gain,wrong_after,within_1sigma_before,within_1sigma_after
0,official,GBM residual,53.54,54.17,"+0.63 [+0.13, +1.14]",0,0.951,0.980
1,nudt101,GBM residual,55.56,55.64,"+0.08 [+0.00, +0.24]",0,0.989,0.993
2,nudt202,GBM residual,57.36,57.48,"+0.12 [-0.24, +0.48]",0,0.984,0.990
3,nudt404,GBM residual,53.43,53.31,"-0.12 [-0.72, +0.36]",4,0.958,0.953
